In [ ]:
import numpy as np
import pandas as pd

from raas.neural_policy import DPAgent
# from raas.discrete_policy import DiscretizedDPAgent
from raas.optimized_discrete_policy import DiscretizedDPAgent
from raas.simulation import Simulator, CustomerGenerator
from raas.hazard_models import ExponentialHazard
from raas.utility_learner import ProjectedVolumeLearner
from raas.degradation_learner import DegradationLearner
from datetime import datetime
from pytz import timezone
from raas.utils import PerfectDegradationLearner
import matplotlib.pyplot as plt

import logging
logging.basicConfig(level=logging.INFO)

import raas.config as config

np.set_printoptions(suppress=True)

from raas.utils import calculate_rolling_rate

In [3]:
usage_exp_hazard_model = ExponentialHazard(lambda_val=config.LAMBDA_VAL)

customer_gen = CustomerGenerator(
    d=config.D,
    context_sampler=config.context_sampler,
    rental_sampler=config.rental_sampler,
    interarrival_sampler=config.interarrival_sampler
)

projected_volume_learner = ProjectedVolumeLearner(
    T=config.NUM_CUSTOMERS, 
    d=config.D, 
    centroid_params=config.centroid_params,
    incentive_constant=config.incentive_constant,
    termination_rule=config.termination_rule,
)

# Instantiate the Simulator with the new parameters
simulator = Simulator(
    d=config.D,
    T=config.NUM_CUSTOMERS,
    
    theta_true=config.THETA_TRUE,
    utility_true=config.UTILITY_TRUE,
    pricing_r=config.PRICING_R,
    
    usage_hazard_model=usage_exp_hazard_model,
    customer_generator=customer_gen,
    projected_volume_learner=projected_volume_learner,  # Use default ProjectedVolumeLearner
    
    mdp_params=config.mdp_params,
    discrete_dp=True,
    policy_type=config.policy_type,
    training_hyperparams=config.training_hyperparams,
    policy_kwargs=config.policy_kwargs,
    policy_update_threshold=100,
    time_normalize=True,
)

In [ ]:
# # Lets you skip utility exploration with perfect u starting point
simulator.projected_volume_learner.centroids.append(config.UTILITY_TRUE)
simulator.projected_volume_learner.is_terminated = True
simulator.seen_breakdowns = 2

degradation_learner = DegradationLearner(d=simulator.d)
degradation_learner.theta = np.ones(config.D) * 0.1
degradation_learner.cum_baseline = lambda x: config.LAMBDA_VAL * x
degradation_learner.inverse_cum_baseline = lambda y: y / config.LAMBDA_VAL
simulator.degradation_learner = degradation_learner

# dp_agent = DPAgent(
#     d=simulator.d,
#     u_hat=config.UTILITY_TRUE,
#     time_normalize=simulator.time_normalize,
#     degradation_learner=simulator.degradation_learner,
#     customer_generator=simulator.customer_generator,
#     params=simulator.mdp_params
# )
# dp_agent.train(**simulator.training_hyperparams)

dp_agent = DiscretizedDPAgent(
    N=config.training_hyperparams['N'], # grid sizes [cum_context, context, duration, active_time]
    max_cumulative_context=config.training_hyperparams['max_cumulative_context'],
    # max_active_time=config.training_hyperparams['max_active_time'],
    u_hat=config.UTILITY_TRUE,
    degradation_learner=degradation_learner,
    customer_generator=customer_gen,
    params=config.mdp_params,
)

# dp_agent._precompute_dynamics(num_samples=50000)
dp_agent.run_value_iteration(100)

simulator.dp_agent = dp_agent
simulator.optimal_policy = dp_agent.get_policy(simulator.policy_type)
simulator.breakdowns_since_last_update = 0 # Reset the counter

In [ ]:
pacific_tz = timezone('America/Los_Angeles')
current_time = datetime.now(pacific_tz).strftime("%Y%m%d_%H%M%S")

# simulator.projected_volume_learner.is_terminated = True
simulation_data = simulator.run(num_customers=config.NUM_CUSTOMERS)
degradation_df = pd.DataFrame(simulator.degradation_history)
simulation_df = pd.DataFrame(simulator.history)

degradation_df.to_csv(f'../data/degradation_data_{current_time}.csv', index=False)
simulation_df.to_csv(f'../data/simulation_data_{current_time}.csv', index=False)
simulator.save(f'../models/simulator_{current_time}')

In [ ]:
degradation_df = pd.DataFrame(simulator.degradation_history)
simulation_df = pd.DataFrame(simulator.history)

degradation_df.to_csv(f'../data/degradation_data_{current_time}.csv', index=False)
simulation_df.to_csv(f'../data/simulation_data_{current_time}.csv', index=False)
simulator.save(f'../models/simulator_{current_time}')

### Convergence of $\hat\theta$

In [ ]:
# simulator = Simulator.load('models/simulator_0914')

history = pd.DataFrame(simulator.history)
degradation_history = pd.DataFrame(simulator.degradation_history)

epsilons = [0.20 * (0.95 ** i) for i in range(len(simulator.theta_updates))]

times = []

for d in simulator.theta_updates:
    idx, theta_hat = d['customer_idx'], d['theta_hat']
    time = history[history.customer_id == idx]['calendar_time'].max()
    times.append(time)
    
# plot L2, and L-inf norms of utility updates
L2_errors = [np.linalg.norm(update['theta_hat'] - config.THETA_TRUE) for update in simulator.theta_updates]
Linf_errors = [np.linalg.norm(update['theta_hat'] - config.THETA_TRUE, ord=np.inf) for update in simulator.theta_updates]

plt.figure(figsize=(12, 6))
plt.plot(times, L2_errors, label='$L_2$ Norm Error', marker='o')
plt.plot(times, Linf_errors, label='$L_\infty$ Norm Error', marker='x')
plt.plot(times, epsilons, label='Exploration Rate (ε)', linestyle='--', color='gray')
# plt.yscale('log')
plt.xlabel('Number of Customers Processed', fontsize=14)
plt.ylabel('Error Norm', fontsize=14)

plt.title('Convergence of $\|\hat{\\theta} - \\theta\|$', fontsize=18)
plt.legend(fontsize=12)
plt.grid(True)
# plt.savefig('../figures/utility_convergence.pdf')
plt.show()

### Convergence of $\hat u$

In [ ]:
simulator.utility_updates

# plot L2, and L-inf norms of utility updates
L2_errors = [np.linalg.norm(update['u_hat'] - config.UTILITY_TRUE) for update in simulator.utility_updates]
Linf_errors = [np.linalg.norm(update['u_hat'] - config.UTILITY_TRUE, ord=np.inf) for update in simulator.utility_updates]

plt.figure(figsize=(12, 6))
plt.plot(L2_errors, label='$L_2$ Norm Error', marker='o')
plt.plot(Linf_errors, label='$L_\infty$ Norm Error', marker='x')
# plt.yscale('log')
plt.xlabel('Number of Customers Processed', fontsize=14)
plt.ylabel('Error Norm', fontsize=14)

plt.title('Convergence of $\|\hat u - u\|$', fontsize=18)
plt.legend(fontsize=12)
plt.grid(True)
# plt.savefig('../figures/utility_convergence.pdf')
plt.show()

### Revenue of Online Learner

In [ ]:
degradation_df = pd.DataFrame(simulator.degradation_history)
simulation_df = pd.DataFrame(simulator.history)

simulation_df['net_profit'] = simulation_df['profit'] + simulation_df['loss']
simulation_df['cumulative_net_profit'] = simulation_df['net_profit'].cumsum()

max_time = 20000

plot_df = simulation_df[
     (simulation_df['calendar_time'] <= max_time)
]

ax = plt.figure(figsize=(10,6))

# plot cumulative profit and loss over time
plt.plot(plot_df['calendar_time'], plot_df['cumulative_net_profit'], label='Cumulative Net Profit')
plt.xlabel('Calendar Time')
plt.ylabel('Cumulative Net Profit')
plt.title('Cumulative Net Profit Over Time')
plt.legend()
plt.grid()
plt.savefig('../figures/cumulative_net_profit_online.pdf')
plt.show()

## Training policy under perfect information

### Revenue of Optimal Policy

In [ ]:
# perfect_degradation_learner, perfect_dpagent, perfect_policy = get_perfect_degradation_learner(
#     sample_size=200000,
#     iterations=300
# )

# perfect_dpagent.save_policy('../models/perfect_discrete_policy.pkl')

In [ ]:
perfect_dpagent = DiscretizedDPAgent.load_policy('../models/perfect_discrete_policy.pkl')

In [ ]:
simulation_df = pd.DataFrame(simulator.history)
perfect_degradation_learner = PerfectDegradationLearner(
    d=config.D, 
    theta_true=config.THETA_TRUE,
    hazard_model=usage_exp_hazard_model,
)

samples = simulator.run_full_exploit(
    100000, 
    perfect_dpagent.get_policy('greedy'), 
    perfect_degradation_learner,
    {'tau': 0.01}
)
samples = pd.DataFrame(samples)

simulation_df['net_profit'] = simulation_df['profit'] + simulation_df['loss']
samples['net_profit'] = samples['profit'] + samples['loss']

simulation_df['cumulative_net_profit'] = simulation_df['net_profit'].cumsum()
samples['cumulative_net_profit'] = samples['net_profit'].cumsum()

samples['netprofit_per_time'] = samples['cumulative_net_profit'] / samples['calendar_time']
simulation_df['netprofit_per_time'] = simulation_df['cumulative_net_profit'] / simulation_df['calendar_time']

In [ ]:
# --- 3. Plot the new rolling profit rate ---

window_duration = 100 # Define the time window for the rolling rate

for df in [simulation_df, samples]:
# for df in [samples]:
    df['net_profit'] = df['profit'] + df['loss']
    # Add the new 'profit_rate' column using our helper function
    df['profit_rate'] = calculate_rolling_rate(df, 'calendar_time', 'net_profit', window_duration)
    
max_time = 5000 # min(simulation_df['calendar_time'].max(), samples['calendar_time'].max())
# max_time = simulation_df['calendar_time'].max()
# samples_plot = samples[(window_duration <= samples['calendar_time']) & (samples['calendar_time'] <= max_time)]
simulations_plot = simulation_df[
    (window_duration <= simulation_df['calendar_time']) &
    (simulation_df['calendar_time'] <= max_time)]

samples_plot = samples[
    (window_duration <= samples['calendar_time']) &
    (samples['calendar_time'] <= max_time)]

plt.figure(figsize=(10, 6))

plt.plot(samples_plot['calendar_time'], samples_plot['profit_rate'], label=f'Optimal Policy (Rolling {window_duration} unit avg)')
plt.plot(simulations_plot['calendar_time'], simulations_plot['profit_rate'], label=f'Online Learning (Rolling {window_duration} unit avg)')

plt.xlabel('Calendar Time')
plt.ylabel('Profit Rate (Profit / Time Unit)')
plt.title(f'Rolling Profit Rate Over Time (Window = {window_duration} time units)')
plt.legend()
plt.grid(True)
plt.show()